# D9：Pandas 索引、条件筛选与排序

2026-10-20。请从上往下运行，每次运行前先预测结果形状。
**Shift + Enter** 运行单元格；**Ctrl + S** 保存。
内核选择 **Python (D Learning)**。独立验收请使用 practice.ipynb。

prices.csv 是虚构的教学样本，不是实际市场行情。每行是一个资产某一天的记录：
record_id（编号）、date（日期）、asset（资产代号）、price（价格）、volume（成交量）。
原表 8 行 5 列，A/B 各四天。price 为教学用价格数值，volume 为教学用成交量数值。


## 1. 读取与设置索引

默认索引为 0～7；record_id 最初是普通列。set_index 生成一个新表，把编号改作索引。
index 是行标签，position 是行所在的位置，二者并不一定相等。
parse_dates 将指定字段读成日期，便于按时间先后排序。


In [ ]:
from pathlib import Path  # Path：文件路径
import pandas as pd  # pandas 简写为 pd

data_dir = Path("D:/Learning/study/d9/data")  # 今天的数据文件夹
raw = pd.read_csv(data_dir / "prices.csv", parse_dates=["date"])  # date 读成日期
df = raw.set_index("record_id")  # df 是 DataFrame 的常见变量名缩写

print(raw.shape)  # (8, 5)，默认索引不计入五个数据列
print(df.shape)  # (8, 4)，record_id 已变成索引
print(df)
df.info()  # 查看 date 的日期类型以及其他字段类型


## 2. 条件一：筛选资产 A

== 表示比较是否相等；= 表示赋值。
一个列与一个值比较，会对每行分别判断，得到布尔 Series。
布尔表示 True（满足）或 False（不满足）；mask 是这种筛选标记的常用变量名。


In [ ]:
mask_a = df["asset"] == "A"  # 给每行标记是否属于资产 A
asset_a = df.loc[mask_a]  # 保留标记为 True 的行

print(mask_a)  # 8 个布尔值，索引仍与 df 对应
print(asset_a)
print(asset_a.shape)  # (4, 4)，编号 101、103、105、107


## 3. 条件二：筛选价格严格超过 103

此时检查所有资产，暂不限制资产代号。> 不包含等于阈值的情况。
没有满足条件的行时，得到空表也是正常结果。


In [ ]:
high_price = df.loc[df["price"] > 103]  # 所有资产中，价格严格大于 103 的行
print(high_price)
print(high_price.shape)  # (6, 4)，编号 102、103、104、106、107、108

empty_result = df.loc[df["price"] > 300]  # 本样本中没有价格大于 300 的记录
print(empty_result.shape)  # (0, 4)，没有行但仍有四列


## 4. 条件三：资产 A，并且价格大于 103

& 表示逐行“并且”，两个比较条件分别放在括号内，避免运算优先级造成错误。
and 不能直接组合这里的两组布尔 Series。
补充：| 表示逐行“或者”；~ 表示逐行取反。


In [ ]:
combined_mask = (df["asset"] == "A") & (df["price"] > 103)  # 两个条件都成立
selected = df.loc[combined_mask]  # 选出同时满足两项条件的记录

print(selected)
print(selected.shape)  # (2, 4)，编号 103、107，价格 105、108


## 5. 打乱行顺序

sample 是抽样；frac=1 表示取全部行；random_state=42 固定随机结果，方便复现。
本例相当于把所有记录洗牌。原来的 df 不变，新顺序放在 shuffled 中。
索引标签会跟着整条记录移动，不会被自动改成新的行位置。


In [ ]:
shuffled = df.sample(frac=1, random_state=42)  # 取全部行并打乱顺序
print(shuffled)
print("打乱后的索引：", shuffled.index.tolist())  # 把索引转成列表，方便观察
print("原表索引：", df.index.tolist())  # 仍然是 101～108


## 6. 按列值排序

sort_values 按数据列的值排序；ascending 表示是否升序，默认 True。
多列排序先比较第一列，相同资产内部再按日期先后排列。
这会先显示 A 的四天，再显示 B 的四天，不是全表仅按日期排序。


In [ ]:
ordered = shuffled.sort_values(
    by=["asset", "date"],  # 第一排序依据是资产，第二依据是日期
    ascending=[True, True],  # 两项都升序：A 在 B 前，日期从早到晚
)
print(ordered)
print(ordered.shape)  # (8, 4)，只是重排行，没有增加或删除记录
print("排序后的索引：", ordered.index.tolist())
# 预计：[101, 103, 105, 107, 102, 104, 106, 108]


## 7. 排序后，标签与位置继续有区别

排序改变的是记录所在的位置，原来的索引标签仍保留。
loc[102] 找编号为 102 的记录；iloc[1] 找当前排列中的第 2 条记录。


In [ ]:
print(ordered.iloc[1])  # 当前位置 1：编号 103 的 A 记录，价格 105
print(ordered.loc[102])  # 标签 102：B 的第一天，价格 200


## 8. sort_index 与 reset_index

sort_index 按索引标签排序；reset_index 重新生成 0～7 的索引，默认保留旧索引为普通列。
本例的记录编号也恰好对应原始顺序，但 sort_index 并不保证恢复任意原始排列。
本例用 reset_index() 保留 record_id；drop=True 则会丢弃旧索引标签。


In [ ]:
by_id = shuffled.sort_index()  # 按索引标签排序，本例是编号从小到大
print("按索引排序：", by_id.index.tolist())  # 101～108

reset_table = ordered.reset_index()  # 旧索引 record_id 变回普通列
print(reset_table)
print(reset_table.shape)  # (8, 5)
print("新索引：", reset_table.index.tolist())  # 0～7


## 9. 新算法题 217：存在重复元素

判断数字列表是否包含重复值；返回 True 或 False。
set（集合）只保留不同的值；空集合写 set()，空的 {} 是字典。
seen 保存此前见过的数字。每次先查是否出现过，未出现才 add 进去。
return False 必须在循环外：只有全部检查完，才能判定没有重复。


In [ ]:
def contains_duplicate(nums):  # 判断输入数字列表是否存在重复值
    seen = set()  # seen：已经见过的数字；set() 创建空集合

    for number in nums:  # 按顺序检查列表中的每个数字
        if number in seen:  # in：判断这个数字是否已在集合中
            return True  # 已经出现过，说明存在重复，立即返回 True
        seen.add(number)  # add：把第一次见到的数字加入集合

    return False  # 全部检查完仍无重复，返回 False；这句位于循环外

print(contains_duplicate([2, 5, 2]))  # True，重复的 2 不需要相邻
print(contains_duplicate([2, 5, 7]))  # False
print(contains_duplicate([7]))  # False，只有一个元素
print(contains_duplicate([1, 1]))  # True


## 10. 独立完成与复习

关闭参考版，完成 practice.ipynb 和 notes.md。
第 217 题参考代码另存于 D:/Learning/study/cs61A/217.py，便于集中复习。
你的独立实现可以另存 217_practice.py，以便与参考版比较。

官方资料：[条件筛选](https://pandas.pydata.org/pandas-docs/version/2.3/user_guide/indexing.html#boolean-indexing)、
[排序](https://pandas.pydata.org/pandas-docs/version/2.3/reference/api/pandas.DataFrame.sort_values.html)、
[重置索引](https://pandas.pydata.org/pandas-docs/version/2.3/reference/api/pandas.DataFrame.reset_index.html)。
